In [2]:
# In your first cell
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import KFold
from google.colab import drive



In [8]:
# Mount Drive and Define Paths
drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/value-vortex/'
DATA_PATH = PROJECT_PATH + 'dataset/'

# Load Data
train_df = pd.read_csv(DATA_PATH + 'train.csv')
test_df = pd.read_csv(DATA_PATH + 'test.csv')

print("Setup Complete. Data is loaded.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup Complete. Data is loaded.


In [10]:
# In your second cell

# 1. Log transform the target
train_df['log_price'] = np.log1p(train_df['price'])


# 2. Create TF-IDF features
tfidf_vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))

X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['catalog_content'])


y_train_log = train_df['log_price'] # Our target series

print("Baseline features and target are ready.")
# === REPLACE THE BOTTOM HALF OF YOUR FEATURE CREATION CELL WITH THIS ===

from scipy.sparse import hstack
from sklearn.preprocessing import StandardScaler

# 1. Load the new features you created (this part is the same)
FEATURES_PATH = PROJECT_PATH + 'features/'
print("Loading new NLP features...")
ipq_train = np.load(FEATURES_PATH + 'ipq_train.npy')
embeddings_train = np.load(FEATURES_PATH + 'train_embeddings_minilm.npy')
print("New features loaded.")

# 2. <<< NEW STEP: Scale the dense features >>>
print("Scaling dense features (IPQ and Embeddings)...")

# --- FIX: ADD THESE TWO MISSING LINES ---
ipq_train_reshaped = ipq_train.reshape(-1, 1)
scaler_ipq = StandardScaler()
# ------------------------------------

scaler_embeddings = StandardScaler()

# Fit the scalers and transform the features
ipq_train_scaled = scaler_ipq.fit_transform(ipq_train_reshaped)
embeddings_train_scaled = scaler_embeddings.fit_transform(embeddings_train)

# 3. Consolidate ALL features into one matrix (using the SCALED versions)
print("Consolidating all features...")
X_train_consolidated = hstack([
    X_train_tfidf,                # Original TF-IDF
    ipq_train_scaled,             # SCALED IPQ feature
    embeddings_train_scaled       # SCALED Sentence embeddings
]).tocsr()

print(f"Final consolidated feature shape: {X_train_consolidated.shape}")

Baseline features and target are ready.
Loading new NLP features...
New features loaded.
Scaling dense features (IPQ and Embeddings)...
Consolidating all features...
Final consolidated feature shape: (75000, 10385)


In [ ]:
# In your third cell

def smape(y_true, y_pred):
    """
    Calculates the Symmetric Mean Absolute Percentage Error (SMAPE).
    """
    numerator = np.abs(y_pred - y_true)
    denominator = (np.abs(y_true) + np.abs(y_pred)) / 2

    # Handle the case where both y_true and y_pred are zero
    # to avoid division by zero.
    ratio = np.where(denominator == 0, 0, numerator / denominator)

    return np.mean(ratio) * 100

print("SMAPE metric function is defined.")

SMAPE metric function is defined.


In [ ]:
# In your fourth cell

# --- Configuration ---
N_SPLITS = 5 # A standard number for K-Fold
MODEL_PARAMS = {
    'random_state': 42,
    # We can add more LightGBM parameters here later
}

# --- Initialization ---
kf = KFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
oof_scores = [] # This will store the SMAPE score for each fold

# --- CV Loop ---
print(f"Starting {N_SPLITS}-Fold Cross-Validation...")

for fold, (train_index, val_index) in enumerate(kf.split(X_train_consolidated, y_train_log)):
    print(f"===== FOLD {fold+1} =====")

    # 1. Split data into training and validation for this fold
    X_train_fold, X_val_fold = X_train_consolidated[train_index], X_train_consolidated[val_index]
    y_train_fold, y_val_fold_log = y_train_log.iloc[train_index], y_train_log.iloc[val_index]

    # 2. Initialize the model (it's important to do this inside the loop)
    model = lgb.LGBMRegressor(**MODEL_PARAMS)

    # 3. Train the model
    model.fit(X_train_fold, y_train_fold)

    # 4. Predict on the validation set
    val_preds_log = model.predict(X_val_fold)

    # 5. Inverse transform predictions and true values to calculate SMAPE
    val_preds = np.expm1(val_preds_log)
    y_val_true = np.expm1(y_val_fold_log)

    # 6. Calculate and store the score for this fold
    fold_score = smape(y_val_true, val_preds)
    oof_scores.append(fold_score)
    print(f"Fold {fold+1} SMAPE: {fold_score:.4f}")

# --- Final Score ---
mean_smape = np.mean(oof_scores)
std_smape = np.std(oof_scores)
print("\n--- CV Results ---")
print(f"Mean SMAPE over {N_SPLITS} folds: {mean_smape:.4f}")
print(f"Standard Deviation: {std_smape:.4f}")

Starting 5-Fold Cross-Validation...
===== FOLD 1 =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 16.727620 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1305355
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 10382
[LightGBM] [Info] Start training from score 2.740904


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 1 SMAPE: 56.6419
===== FOLD 2 =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 16.470563 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1303162
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 10381
[LightGBM] [Info] Start training from score 2.738173


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 2 SMAPE: 55.8701
===== FOLD 3 =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 17.530449 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1305708
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 10382
[LightGBM] [Info] Start training from score 2.741725


...

Fold 3 SMAPE: 56.2152
===== FOLD 4 =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 16.335760 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1304087
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 10380
[LightGBM] [Info] Start training from score 2.737836


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 4 SMAPE: 55.1870
===== FOLD 5 =====
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 17.487087 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1304180
[LightGBM] [Info] Number of data points in the train set: 60000, number of used features: 10382
[LightGBM] [Info] Start training from score 2.737449


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


Fold 5 SMAPE: 56.1999

--- CV Results ---
Mean SMAPE over 5 folds: 56.0228
Standard Deviation: 0.4844


In [4]:
!pip install optuna -q
import optuna

# --- 1. Define the Objective Function ---
# This function takes a 'trial' object, suggests hyperparameters,
# runs our cross-validation, and returns the score for Optuna to minimize.

def objective(trial):
    # Define the search space for the hyperparameters
    params = {
        'objective': 'regression_l1', # MAE is closer to SMAPE than MSE
        'metric': 'mae',
        'random_state': 42,
        'n_estimators': 1000, # We'll use more estimators and early stopping
        'verbosity': -1,

        # Parameters to be tuned by Optuna
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'feature_fraction': trial.suggest_float('feature_fraction', 0.4, 1.0),
        'bagging_fraction': trial.suggest_float('bagging_fraction', 0.4, 1.0),
        'bagging_freq': trial.suggest_int('bagging_freq', 1, 7),
        'lambda_l1': trial.suggest_float('lambda_l1', 1e-8, 10.0, log=True),
        'lambda_l2': trial.suggest_float('lambda_l2', 1e-8, 10.0, log=True),
    }

    oof_scores = []

    # Use the same KFold split as before
    for fold, (train_index, val_index) in enumerate(kf.split(X_train_consolidated, y_train_log)):
        X_train_fold, X_val_fold = X_train_consolidated[train_index], X_train_consolidated[val_index]
        y_train_fold, y_val_fold_log = y_train_log.iloc[train_index], y_train_log.iloc[val_index]

        # Use a callback for early stopping
        early_stopping_callback = lgb.early_stopping(100, verbose=False)

        model = lgb.LGBMRegressor(**params)
        model.fit(X_train_fold, y_train_fold,
                  eval_set=[(X_val_fold, y_val_fold_log)],
                  eval_metric='mae',
                  callbacks=[early_stopping_callback])

        val_preds_log = model.predict(X_val_fold)
        val_preds = np.expm1(val_preds_log)
        y_val_true = np.expm1(y_val_fold_log)

        fold_score = smape(y_val_true, val_preds)
        oof_scores.append(fold_score)

    return np.mean(oof_scores)


# --- 2. Run the Optuna Study ---
# We want to MINIMIZE the SMAPE score, so the direction is 'minimize'
study = optuna.create_study(direction='minimize')

# Let's run it for 50 trials to start. This might take an hour or more.
study.optimize(objective, n_trials=50)

# --- 3. Print the Best Results ---
print("\n--- Optuna Study Complete ---")
print("Number of finished trials: ", len(study.trials))
print("Best trial:")
trial = study.best_trial
print("  Value (SMAPE): ", trial.value)
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.9/400.9 kB 8.0 MB/s eta 0:00:00


[I 2025-10-13 06:04:50,937] A new study created in memory with name: no-name-f8654624-9fb7-47b1-a7e6-e6b0fb8e1041
[W 2025-10-13 06:04:50,942] Trial 0 failed with parameters: {'learning_rate': 0.08831325195812591, 'num_leaves': 34, 'max_depth': 11, 'feature_fraction': 0.8300323852268283, 'bagging_fraction': 0.614976753380781, 'bagging_freq': 7, 'lambda_l1': 0.00019604178477854823, 'lambda_l2': 0.00851264992322722} because of the following error: NameError("name 'kf' is not defined").
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/optuna/study/_optimize.py", line 201, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "/tmp/ipython-input-1383134797.py", line 31, in objective
    for fold, (train_index, val_index) in enumerate(kf.split(X_train_consolidated, y_train_log)):
                                                    ^^
NameError: name 'kf' is not defined
[W 2025-10-13 06:04:50,944] Trial 0 failed with value N

NameError: name 'kf' is not defined

In [ ]:
# Run this in a new cell after stopping the study
print("--- Best Optuna Results ---")
best_trial = study.best_trial
best_params = best_trial.params

print(f"Best CV Score (SMAPE): {best_trial.value:.4f}")
print("Best Hyperparameters:")
for key, value in best_params.items():
    print(f"    '{key}': {value},")

--- Best Optuna Results ---
Best CV Score (SMAPE): 52.5733
Best Hyperparameters:
    'learning_rate': 0.03965371679098294,
    'num_leaves': 208,
    'max_depth': 12,
    'feature_fraction': 0.7860765673620694,
    'bagging_fraction': 0.9854512237316982,
    'bagging_freq': 4,
    'lambda_l1': 0.061599106746932204,
    'lambda_l2': 0.0333393312021117,


In [11]:
import joblib
import os

# Paste the best parameters from your Optuna study here
best_params = {
    'learning_rate': 0.03965371679098294,
    'num_leaves': 208,
    'max_depth': 12,
    'feature_fraction': 0.7860765673620694,
    'bagging_fraction': 0.9854512237316982,
    'bagging_freq': 4,
    'lambda_l1': 0.061599106746932204,
    'lambda_l2': 0.0333393312021117
}

# Add back the non-tuned parameters
best_params['objective'] = 'regression_l1'
best_params['metric'] = 'mae'
best_params['random_state'] = 42
best_params['n_estimators'] = 2000 # Use a higher number for the final model

print("Training the final, tuned text-only model on 100% of the data...")
final_text_model = lgb.LGBMRegressor(**best_params)
final_text_model.fit(X_train_consolidated, y_train_log)

# Save this powerful model for later
MODELS_PATH = PROJECT_PATH + 'models/'
os.makedirs(MODELS_PATH, exist_ok=True)
joblib.dump(final_text_model, MODELS_PATH + 'best_text_model.pkl')
print("Best text-only model has been trained and saved!")

Training the final, tuned text-only model on 100% of the data...
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.7860765673620694, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7860765673620694
[LightGBM] [Warning] lambda_l2 is set=0.0333393312021117, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0333393312021117
[LightGBM] [Warning] lambda_l1 is set=0.061599106746932204, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.061599106746932204
[LightGBM] [Warning] bagging_fraction is set=0.9854512237316982, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9854512237316982
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.7860765673620694, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7860765673

In [12]:
print("Creating consolidated features for the test set...")

# Load the raw test features
ipq_test = np.load(FEATURES_PATH + 'ipq_test.npy')
embeddings_test = np.load(FEATURES_PATH + 'test_embeddings_minilm.npy')

# IMPORTANT: Use the scalers that were already FITTED on the training data
ipq_test_scaled = scaler_ipq.transform(ipq_test.reshape(-1, 1))
embeddings_test_scaled = scaler_embeddings.transform(embeddings_test)

# Create TF-IDF features for the test set
X_test_tfidf = tfidf_vectorizer.transform(test_df['catalog_content'])

# Consolidate all test features
X_test_consolidated = hstack([
    X_test_tfidf,
    ipq_test_scaled,
    embeddings_test_scaled
]).tocsr()

print("Consolidated test features created.")


Creating consolidated features for the test set...
Consolidated test features created.


In [13]:
print("Generating predictions with the tuned model...")
predictions_log = final_text_model.predict(X_test_consolidated)
predictions = np.expm1(predictions_log)
predictions[predictions < 0] = 0 # Enforce positive prices

# Create and save the new submission file
submission_df = pd.DataFrame({'sample_id': test_df['sample_id'], 'price': predictions})
SUBMISSIONS_PATH = PROJECT_PATH + 'submissions/'
os.makedirs(SUBMISSIONS_PATH, exist_ok=True)
submission_df.to_csv(SUBMISSIONS_PATH + 'submission_text_tuned.csv', index=False)

print(f"New submission file 'submission_text_tuned.csv' has been created!")

Generating predictions with the tuned model...
[LightGBM] [Warning] bagging_freq is set=4, subsample_freq=0 will be ignored. Current value: bagging_freq=4
[LightGBM] [Warning] feature_fraction is set=0.7860765673620694, colsample_bytree=1.0 will be ignored. Current value: feature_fraction=0.7860765673620694
[LightGBM] [Warning] lambda_l2 is set=0.0333393312021117, reg_lambda=0.0 will be ignored. Current value: lambda_l2=0.0333393312021117
[LightGBM] [Warning] lambda_l1 is set=0.061599106746932204, reg_alpha=0.0 will be ignored. Current value: lambda_l1=0.061599106746932204
[LightGBM] [Warning] bagging_fraction is set=0.9854512237316982, subsample=1.0 will be ignored. Current value: bagging_fraction=0.9854512237316982


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(


New submission file 'submission_text_tuned.csv' has been created!
